In [16]:
import pandas as pd
import numpy as np

## Import slimmed down Gold-Standard

In [17]:
gs = pd.read_csv("../gs_slim.csv")
gs['year'] = gs['year'].astype(str) #Correction needed for e.g. fiscal years "FY 2021/2022"

## Import all 3x2=6 Extractions

In [18]:
think1 = pd.read_csv("./PipelineB-Answers/1st_Qwen3-VL-32B-Thinking.csv")
think2 = pd.read_csv("./PipelineB-Answers/2nd_Qwen3-VL-32B-Thinking.csv")

moe1   = pd.read_csv("./PipelineB-Answers/1st_Qwen3-VL-30B-A3B-Thinking.csv")
moe2   = pd.read_csv("./PipelineB-Answers/2nd_Qwen3-VL-30B-A3B-Thinking.csv")

instr1 = pd.read_csv("./PipelineB-Answers/1st_Qwen3-VL-32B-Instruct.csv")
instr2 = pd.read_csv("./PipelineB-Answers/2nd_Qwen3-VL-32B-Instruct.csv")

In [19]:
dfs = [think1, think2, moe1, moe2, instr1, instr2]

scope_mapping = {
    'scope_1': "1",
    'scope_2_location_based': "2lb",
    'scope_2_market_based' : "2mb",
    'scope_3' : "3"
}

for df in dfs:
    df['scope'] = df['scope'].replace(scope_mapping)

Checking if all `report_names` are the same

In [20]:
print("Think ok:", all(think1["report_name"].unique() == think2["report_name"].unique()))
print("Moe   ok:", all(think1["report_name"].unique() == think2["report_name"].unique()))
print("Instr ok:", all(think1["report_name"].unique() == think2["report_name"].unique()))

Think ok: True
Moe   ok: True
Instr ok: True


Checking if all 'report_names' from the extractions are exactly matched in the GoldStandard

In [21]:
think1["report_name"].isin(gs["report_name"]).all()

np.True_

## Merging Extraction Values and Gold-Standard

In [26]:
merge_on = ["report_name", "scope", "year"]
agg_cols = ["value", "unit", "label"]

df_to_merge = [
    (think1, "_t1"),
    (think2, "_t2"),
    (moe1, "_m1"),
    (moe2, "_m2"),
    (instr1, "_i1"),
    (instr2, "_i2")
]

In [ ]:
for df, sf in df_to_merge:
    dups = df.duplicated(subset=merge_on).sum()
    print(f"{sf}: {dups} doppelte Keys")


_t1: 147 doppelte Keys
_t2: 160 doppelte Keys
_m1: 190 doppelte Keys
_m2: 148 doppelte Keys
_i1: 202 doppelte Keys
_i2: 300 doppelte Keys


=> That's way I need not a normal merge, but consolidate every extracted value, unit, and label from every dataframe (think1, think2, moe1, ...) value into a list.

Then I can validate if the gs-value is extracted or not

In [ ]:
# merged = gs.copy() # Setting a starting point for the loop. Everything has to be mapped to the Gold-Standard

# for df, sf in df_to_merge:
#     merged = pd.merge(
#         merged,
#         df,
#         on  = merge_on,
#         how = "left",
#         suffixes =["", sf]
#     )

In [28]:
merged = gs.copy() # Setting a starting point for the loop. Everything has to be mapped to the Gold-Standard

for df, sf in df_to_merge:
    agg = (
        df.groupby(merge_on)[agg_cols]
        .agg(list)
        .reset_index()
        .rename(columns={col: f"{col}{sf}" for col in agg_cols})
    )
    merged = pd.merge(merged, agg, on=merge_on, how="left")
    
print(f"Zeilen: {len(merged)} (GS hatte {len(gs)})")
print(f"Spalten: {merged.columns.tolist()}")

Zeilen: 5631 (GS hatte 5631)
Spalten: ['report_name', 'year', 'scope', 'page', 'value', 'unit', 'unit_normalized', 'label', 'status', 'scopes_present', 'value_t1', 'unit_t1', 'label_t1', 'value_t2', 'unit_t2', 'label_t2', 'value_m1', 'unit_m1', 'label_m1', 'value_m2', 'unit_m2', 'label_m2', 'value_i1', 'unit_i1', 'label_i1', 'value_i2', 'unit_i2', 'label_i2']


In [14]:
for df, sf in df_to_merge:
    dups_mask = df.duplicated(subset=merge_on, keep=False)
    print(f"\n=== {sf} ===")
    display(df[dups_mask].sort_values(merge_on).head(20))


=== _t1 ===


,report_name,scope,year,value,unit,label
1,Allianz_2022_report,1,2020,28714.0,t CO2e,Scope 1 – Direct GHG emissions
2,Allianz_2022_report,1,2020,29.0,ktCO2eq,Scope 1
3,Allianz_2022_report,1,2021,28699.0,t CO2e,Scope 1 – Direct GHG emissions
4,Allianz_2022_report,1,2021,29.0,ktCO2eq,Scope 1
5,Allianz_2022_report,1,2022,30953.0,t CO2e,Scope 1 – Direct GHG emissions
6,Allianz_2022_report,1,2022,31.0,ktCO2eq,Scope 1
8,Allianz_2022_report,2mb,2020,100722.0,t CO2e,Scope 2 – Indirect GHG emissions (market based)
9,Allianz_2022_report,2mb,2020,101.0,ktCO2eq,Scope 2 – market based
10,Allianz_2022_report,2mb,2021,54689.0,t CO2e,Scope 2 – Indirect GHG emissions (market based)
11,Allianz_2022_report,2mb,2021,55.0,ktCO2eq,Scope 2 – market based



=== _t2 ===


,report_name,scope,year,value,unit,label
1,Allianz_2022_report,1,2020,28714.0,t CO2e,Direct GHG emissions
2,Allianz_2022_report,1,2020,29.0,ktCO2eq,Operational (Figure 2b)
3,Allianz_2022_report,1,2021,28699.0,t CO2e,Direct GHG emissions
4,Allianz_2022_report,1,2021,29.0,ktCO2eq,Operational (Figure 2b)
5,Allianz_2022_report,1,2022,30953.0,t CO2e,Direct GHG emissions
6,Allianz_2022_report,1,2022,31.0,ktCO2eq,Operational (Figure 2b)
7,Allianz_2022_report,2mb,2020,100722.0,t CO2e,Indirect GHG emissions (market based)
8,Allianz_2022_report,2mb,2020,101.0,ktCO2eq,Operational (Figure 2b)
9,Allianz_2022_report,2mb,2021,54689.0,t CO2e,Indirect GHG emissions (market based)
10,Allianz_2022_report,2mb,2021,55.0,ktCO2eq,Operational (Figure 2b)



=== _m1 ===


...

,report_name,scope,year,value,unit,label
10,Allianz_2022_report,3,2021,55359.0,t CO2e,Operational
11,Allianz_2022_report,3,2021,29.0,mn t CO2e,Portfolio
12,Allianz_2022_report,3,2022,92467.0,t CO2e,Operational
13,Allianz_2022_report,3,2022,46.3,mn t CO2e,Portfolio
29,Daimler_2020_report,3,2020,8.1,tons per vehicle,Scope 3 Production: Procured goods and services
30,Daimler_2020_report,3,2020,0.8,tons per vehicle,Scope 1 and 2: Mercedes-Benz Cars production
31,Daimler_2020_report,3,2020,1.0,tons per vehicle,Scope 3 Logistics: Transport and distribution ...
32,Daimler_2020_report,3,2020,5.6,tons per vehicle,Scope 3 Use phase: Fuel and electricity produc...
33,Daimler_2020_report,3,2020,33.7,tons per vehicle,Scope 3 Use phase: Vehicle operation (tank-to-...
34,Daimler_2020_report,3,2020,0.4,tons per vehicle,Scope 3 End of Life: Recycling and waste disposal



=== _i1 ===


,report_name,scope,year,value,unit,label
1,Allianz_2022_report,1,2020,29,ktCO₂eq,Scope 1 from Figure 2b
2,Allianz_2022_report,1,2020,28714,t CO₂e,Scope 1 – Direct GHG emissions
3,Allianz_2022_report,1,2021,29,ktCO₂eq,Scope 1 from Figure 2b
4,Allianz_2022_report,1,2021,28699,t CO₂e,Scope 1 – Direct GHG emissions
5,Allianz_2022_report,1,2022,31,ktCO₂eq,Scope 1 from Figure 2b
6,Allianz_2022_report,1,2022,30953,t CO₂e,Scope 1 – Direct GHG emissions
8,Allianz_2022_report,2mb,2020,101,ktCO₂eq,Scope 2 market based from Figure 2b
9,Allianz_2022_report,2mb,2020,100722,t CO₂e,Scope 2 – Indirect GHG emissions (market based)
10,Allianz_2022_report,2mb,2021,55,ktCO₂eq,Scope 2 market based from Figure 2b
11,Allianz_2022_report,2mb,2021,54689,t CO₂e,Scope 2 – Indirect GHG emissions (market based)



=== _i2 ===


,report_name,scope,year,value,unit,label
12,Daimler_2020_report,1,2016,1056.0,"in 1,000 t",CO₂ direct (Scope 1)
13,Daimler_2020_report,1,2016,245.0,kg/vehicle,Cars - CO₂ direct (Scope 1)
14,Daimler_2020_report,1,2016,746.0,kg/vehicle,Trucks - CO₂ direct (Scope 1)
15,Daimler_2020_report,1,2016,372.0,kg/vehicle,Vans - CO₂ direct (Scope 1)
16,Daimler_2020_report,1,2016,1408.0,kg/vehicle,Buses - CO₂ direct (Scope 1)
17,Daimler_2020_report,1,2017,1192.0,"in 1,000 t",CO₂ direct (Scope 1)
18,Daimler_2020_report,1,2017,250.0,kg/vehicle,Cars - CO₂ direct (Scope 1)
19,Daimler_2020_report,1,2017,663.0,kg/vehicle,Trucks - CO₂ direct (Scope 1)
20,Daimler_2020_report,1,2017,340.0,kg/vehicle,Vans - CO₂ direct (Scope 1)
21,Daimler_2020_report,1,2017,1177.0,kg/vehicle,Buses - CO₂ direct (Scope 1)


In [ ]:
merged.to_csv("gs_extractions_raw.csv", index=False)

In [12]:
merged.info()

<class 'pandas.DataFrame'>
RangeIndex: 17936465 entries, 0 to 17936464
Data columns (total 28 columns):
 #   Column           Dtype  
---  ------           -----  
 0   report_name      str    
 1   year             str    
 2   scope            str    
 3   page             str    
 4   value            float64
 5   unit             str    
 6   unit_normalized  str    
 7   label            str    
 8   status           str    
 9   scopes_present   str    
 10  value_t1         float64
 11  unit_t1          str    
 12  label_t1         str    
 13  value_t2         float64
 14  unit_t2          str    
 15  label_t2         str    
 16  value_m1         float64
 17  unit_m1          str    
 18  label_m1         str    
 19  value_m2         float64
 20  unit_m2          str    
 21  label_m2         str    
 22  value_i1         str    
 23  unit_i1          str    
 24  label_i1         str    
 25  value_i2         float64
 26  unit_i2          str    
 27  label_i2         str 